# TODO

Make a tool calling class to manage this.

Include a basic panel app that shows the text response to the user and allows accept or reject for each step


# Motivation

Develop some standard system prompt builder tools to support better summarization in a context dependent way.

The vision:
- have an llm read an agenda or overview of the meeting and use some function calling to build a custom summarization prompt
    - transcript meeting
    - reddit or chat thread meeting
    - personal notes meeting
- use summarization prompt to generate summary of meeting
- standardize some report outputs and note style
- create an evals pipelione for summarization
    - take some good summaries and use for evals


## Reference system prompt

https://github.com/saoudrizwan/claude-dev/blob/main/src/ClaudeDev.ts

In [ ]:
import datetime

tools = [
    {
        "name": "list_files",
        "description": "List all files and directories at the top level of the specified directory. Use this to understand the contents and structure of a directory by examining file names and extensions. This information can guide decision-making on which files to process or which subdirectories to explore further. To investigate subdirectories, call this tool again with the path of the subdirectory.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                    "description": "The path of the directory to list contents for.",
                },
            },
            "required": ["path"],
        },
    },
    {
        "name": "read_file",
        "description": "Read the contents of a file at the specified path. Use this when you need to examine the contents of an existing file, for example to analyze code, review text files, or extract information from configuration files. Be aware that this tool may not be suitable for very large files or binary files, as it returns the raw content as a string.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                    "description": "The path of the file to read.",
                },
            },
            "required": ["path"],
        },
    },
    {
        "name": "write_to_report",
        "description": "Write content to a markdown file at the specified path. If the file exists, only the necessary changes will be applied. If the file doesn't exist, it will be created. Always provide the full intended content of the file. This tool will automatically create any directories needed to write the file.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                    "description": "The path of the file to write to.",
                },
                "content": {
                    "type": "string",
                    "description": "The full content to write to the file",
                },
            },
            "required": ["path", "content"],
        },
    },
    {
        "name": "ask_followup_question",
        "description": "Ask the user a question to gather additional information needed to complete the task. This tool should be used when you encounter ambiguities, need clarification, or require more details to proceed effectively. It allows for interactive problem-solving by enabling direct communication with the user. Use this tool judiciously to maintain a balance between gathering necessary information and avoiding excessive back-and-forth.",
        "input_schema": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "The question to ask the user. This should be a clear, specific question that addresses the information you need.",
                },
            },
            "required": ["question"],
        },
    },
    {
        "name": "attempt_completion",
        "description": "Once you've completed the task, use this tool to present the result to the user. They may respond with feedback if they are not satisfied with the result, which you can use to make improvements and try again.",
        "input_schema": {
            "type": "object",
            "properties": {
                "result": {
                    "type": "string",
                    "description": "The result of the task. Formulate this result in a way that is final and does not require further input from the user. Don't end your result with questions or offers for further assistance.",
                },
            },
            "required": ["result"],
        },
    },
]

def get_system_prompt(project_dir:str='./', current_dir:str='./')->str:
    operating_system = 'Windows Subsytem For Linux 2'
    current_time = datetime.datetime.now().isoformat()

    system_prompt = rf"""
    
    You are Claude Summarizer, a highly skilled notetaker and distiller of knowledge from many scientific and technical domains. You take rambling and poorly organized notes, transcripts, and conversations then refactor them into sensible concise and high impact reports.
    
    ====
     
    CAPABILITIES
    
    - You can read and analyze notes in various formats, and can write clean, efficient, and well-documented reports.
    - You can understand complex conversations that may jump from topic to topic and providing detailed explanations, offering architectural insights and information about the key concepts and speakers.
    - You have access to tools that let you execute basic text commands on the user's computer, list files in a directory, read files, write reports, and ask follow-up questions. These tools help you effectively accomplish a wide range of summarization tasks, such as readiong code, reading transcripts or markdown files, understanding the current state of a project, writing reports, and much more.
        - For example, when asked to read notes and summarize you might use the list_files and read_file tools to examine the contents of relevant files, analyze the notes and text and ask the user for clarification on who was at the meeting, then use the write_to_report tool to create a summary report.
    - The write_to_report tool lets you create or update the summary report on the user's computer and should be used whenever you feel you understand the prohject sufficiently well to provide a useful summary of the user's project notes. When you need to execute a write_to_report command, you must provide a clear explanation of why you think you are ready to update or create a summary.
    
    ====
    
    RULES
    - Unless otherwise specified by the user, you MUST accomplish your task within the following directory: '{project_dir}'
    - Your current working directory is '{current_dir}', and you cannot \`cd\` into a different directory to complete a task. You are stuck operating from '{current_dir}', so be sure to pass in the appropriate 'path' parameter when using tools that require a path.
    - Always read a file before editing it if you are missing content. This will help you understand the context and make informed changes.
    - When editing files, always provide the complete file content in your response, regardless of the extent of changes. The system handles diff generation automatically generally by opverwriting the contents with the updates.
    - Before using the write_to_report tool, you must first think about the System Information context provided by the user to understand their environment and tailor your summary report to ensure they are compatible with the user's system and goals.
    - When starting a new summary task, unless the user specifies otherwise, read all the text files in the '{project_dir}' to uinderstand the context better. Use appropriate file paths when writing reports, as the write_to_report tool will automatically create any necessary directories. Structure the report logically, adhering to best practices for the summaryu and notes being written (e.g. overview section, attendees, table of contents/agenda, topics, action items, etc.). Unless otherwise specified, reports should provide context for a stakeholder who is not present for the meeting and help them understand the critical points and get up to speed quickly.
    - You must try to use multiple tools in one request when possible. For example if you were to summarize a project with two files, you would use the read_file tool multiple times to look at several key files and understand their content at once. This will help you accomplish the user's task more efficiently.
    - Be sure to consider the type of notes or text being summarized (e.g. chat conversation, transcript, etc.) when determining the appropriate structure of the report. 
    - Do not ask for more information than necessary. The user may provide feedback, which you can use to make improvements and try again.
    - You are only allowed to ask the user questions using the ask_followup_question tool. Use this tool only when you need additional details to complete a task, and be sure to use a clear and concise question that will help you move forward with the task.
    - Your goal is to try to accomplish the user's task, NOT engage in a back and forth conversation.
    - NEVER end completion_attempt with a question or request to engage in further conversation! Formulate the end of your result in a way that is final and does not require further input from the user. 
    - NEVER start your responses with affirmations like "Certaintly", "Okay", "Sure", "Great", etc. You should NOT be conversational in your responses, but rather direct and to the point.
    
    ====
    
    OBJECTIVE
    
    You accomplish a given summarization task iteratively, breaking it down into clear steps and working through them methodically.
    
    1. Analyze the user's task and set clear, achievable goals to accomplish it. Prioritize these goals in a logical order.
    2. Work through these goals sequentially, utilizing available tools as necessary. Each goal should correspond to a distinct step in your problem-solving process.
    3. Remember, you have extensive capabilities with access to a wide range of tools that can be used in powerful and clever ways as necessary to accomplish each goal. Before calling a tool, do some analysis within <thinking></thinking> tags. First, think about which of the provided tools is the relevant tool to answer the user's request. Second, go through each of the required parameters of the relevant tool and determine if the user has directly provided or given enough information to infer a value. When deciding if the parameter can be inferred, carefully consider all the context to see if it supports a specific value. If all of the required parameters are present or can be reasonably inferred, close the thinking tag and proceed with the tool call. BUT, if one of the values for a required parameter is missing, DO NOT invoke the function (not even with fillers for the missing params) and instead, ask the user to provide the missing parameters using the ask_followup_question tool. DO NOT ask for more information on optional parameters if it is not provided.
    4. The user may provide feedback, which you can use to make improvements and try again. But DO NOT continue in pointless back and forth conversations, i.e. don't end your responses with questions or offers for further assistance.
    
    ====
    
    SYSTEM INFORMATION
    
    Current time: {current_time}
    Operating System: {operating_system}
    """
    return system_prompt

In [ ]:
import json
import pathlib


def read_file(file_path: str) -> str:
    try:
        content = pathlib.Path(file_path).read_text(encoding="utf-8")
        return content
    except Exception as error:
        error_string = f"Error reading file: {json.dumps(str(error))}"
        return error_string

def list_files(path: str) -> str:
    absolute_path = pathlib.Path(path).resolve()
    root = pathlib.Path(absolute_path.anchor)
    is_root = absolute_path == root

    if is_root:
        return str(root)

    try:
        # Use Path.glob() instead of the glob library
        entries = [str(entry.relative_to(absolute_path)) + ('/' if entry.is_dir() else '')
                   for entry in absolute_path.glob('*')]

        # Sort entries to put directories first, then files
        entries.sort(key=lambda x: (not x.endswith('/'), x.lower()))

        result = '\n'.join(entries[:500])  # truncate to 500 entries
        return result

    except Exception as error:
        error_string = f"Error listing files and directories: {json.dumps(str(error))}"
        return error_string

def ask_followup_question(question: str) -> str:
    #asks the user for feedback
    user_response = 'merged_transcript.json' #proxy UI interaction with User
    answer = f"User's response:\n'{user_response}'"
    return answer

def attempt_completion(result: str) -> str:
    result_to_send = result
    #asks the user for feedback
    feedback_text = "The summary is not detailed enough" #proxy a UI interaction
    return f'The user is not pleased with the results. Use the feedback they provided to successfully complete the task, and then attempt completion again.\nUser\'s feedback:\n"{feedback_text}"'

def write_to_report(path: str, content: str) -> dict[str, str|bool]:
    try:
        # Convert the path to a Path object
        file_path = pathlib.Path(path)

        # Create parent directories if they don't exist
        file_path.parent.mkdir(parents=True, exist_ok=True)

        # Check if the file exists and read its content if it does
        existing_content = ""
        if file_path.exists():
            existing_content = file_path.read_text(encoding="utf-8")

        # If the content is different, write the new content
        if content != existing_content:
            file_path.write_text(content, encoding="utf-8")
            changed = True
        else:
            changed = False

        # Prepare the result to be sent to Claude
        result = {
            "tool": "write_to_report",
            "path": str(file_path),
            "content": content,
            "changed": changed
        }

        return {
            "success": True,
            "message": f"Content {'updated' if changed else 'unchanged'} in {file_path}",
            "changed": changed
        }

    except Exception as error:
        error_message = f"Error writing to report: {str(error)}"
        return {"success": False, "message": error_message}

def get_tool(tool_name:str, tool_input):
    match tool_name:
        case "list_files":
            return list_files
        case "read_file":
            return read_file
        case "write_to_report":
            return write_to_report
        case "ask_followup_question":
            return ask_followup_question
        case "attempt_completion":
            return attempt_completion
        case _:
            return f"Unknown tool: {tool_name}"
    return f"Unknown tool: {tool_name}"

In [ ]:
import os

import anthropic
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

# Initial system setup
project_dir = "./summary_files"
current_dir = "./"
system_prompt = get_system_prompt(project_dir, current_dir)

user_prompt = "Task: summarize the notes and provide me a report to share with important stakehodlers who could not attned."


#Agent loop
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
messages = []
#initial ask
messages.append({"role": "user", "content": user_prompt})
response = client.messages.create(
    model="claude-3-haiku-20240307", #"claude-3-5-sonnet-20240620",
    max_tokens=4096,
    system = system_prompt,
    tools=tools,
    messages=messages,
)
print(response.content[0].text)
claude_message = response.content[0].text
messages.append({"role": "assistant", "content": claude_message}) #save

In [ ]:
#tool calling
call_tool = True if response.stop_reason=='tool_use' else False
if call_tool:
    tool_calls = []
    tool_answers = []
    for contentblock in response.content:
        if contentblock.type=='text':
            continue
        elif contentblock.type=='tool_use':
            print("---- Tool Call Requested ----")
            tool_calls.append(contentblock)
            tool_name = contentblock.name
            tool_input = contentblock.input
            tool_use_id = contentblock.id
            print(f"- {tool_name}")
            print(f"- {tool_use_id}")
            print(f"- {tool_input}")
            tool_fn = get_tool(tool_name, tool_input)
            tool_result = tool_fn(**tool_input)
            #parse answer
            tool_answer = {
                "type": "tool_result",
                "tool_use_id": tool_use_id,
                "content": tool_result,
            }
            tool_answers.append(tool_answer)
            print(f"- {tool_result}")
    #compile all answers
    messages.append({"role": "user", "content": tool_answers})


In [ ]:
tool_answers

## Check the tool calling API

In [ ]:
#answer claude
response = client.messages.create(
    model="claude-3-haiku-20240307", #"claude-3-5-sonnet-20240620",
    max_tokens=4096,
    system = system_prompt,
    tools=tools,
    messages=messages,
)
print(response.content[0].text)
claude_message = response.content[0].text
messages.append({"role": "assistant", "content": claude_message}) #save

In [ ]:
response.model_dump()